# Detection / YOLO GC10-DET Evidence Notebook

## Purpose and source-of-truth warning

This notebook presents the finalized YOLO / Detection frontend data-contract bundle for reviewer-facing evidence. It is an evidence/presentation layer only.

The notebook does not train models, does not recompute metrics, does not update registries, and does not create artifacts. The source of truth remains the governed artifacts, inventories, registries, and frontend data-contract JSON files under `artifacts/frontend/detection/yolo_train_v0_2_0/`.

Safety boundary: YOLO / Detection evidence layer is COMPLETE, but this notebook makes no production-ready claim and no deployment-safe claim. This is not real frontend/UI, not API, and not deployment approval.


## 1. Runtime and path setup

The next cell locates the repository root and defines the Detection frontend bundle path. It reads no model outputs yet and performs no computation beyond path setup.

Expected input: the repository root containing `artifacts/frontend/detection/yolo_train_v0_2_0/`.

Expected output: printed repository root and bundle directory. Failure means the notebook is not being run from, or near, the repository checkout.


In [ ]:
from pathlib import Path
import json
import platform
import sys

PROJECT_MARKERS = ["artifacts", "docs", "notebooks", "scripts"]

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in PROJECT_MARKERS):
            return candidate
    raise FileNotFoundError("Could not locate repository root from the current notebook path.")

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
BUNDLE_DIR = REPO_ROOT / "artifacts/frontend/detection/yolo_train_v0_2_0"

print("python_version:", sys.version.split()[0])
print("platform:", platform.platform())
print("repo_root:", REPO_ROOT)
print("bundle_dir:", BUNDLE_DIR)


## 2. Detection bundle file list

The next cell defines the exact frontend bundle JSON files expected for this evidence presentation. It does not read or validate the files yet.

Expected input: no file reads; this is the expected contract.

Expected output: the expected file list. Failure would indicate a notebook editing problem, not a model or artifact problem.


In [ ]:
EXPECTED_BUNDLE_FILES = [
    "detection_overview.json",
    "detection_model_metadata.json",
    "detection_metric_cards.json",
    "detection_confidence_chart.json",
    "detection_class_summary.json",
    "detection_sample_gallery.json",
    "detection_artifact_lineage.json",
    "detection_quality_decision_summary.json",
    "frontend_detection_recommendation.json",
    "frontend_bundle_manifest.json",
]

for name in EXPECTED_BUNDLE_FILES:
    print(name)


## 3. Load and validate frontend bundle files

The next cell reads the 10 JSON files from `artifacts/frontend/detection/yolo_train_v0_2_0/`, validates that each expected file exists, and validates that `frontend_bundle_manifest.json` lists the same files.

Expected input: the finalized Detection frontend bundle JSON files.

Expected output: a compact validation summary. Failure means the local frontend data-contract bundle is missing, malformed, or inconsistent with its manifest.


In [ ]:
missing_files = [name for name in EXPECTED_BUNDLE_FILES if not (BUNDLE_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing Detection frontend bundle files: {missing_files}")

bundle = {}
for name in EXPECTED_BUNDLE_FILES:
    path = BUNDLE_DIR / name
    with path.open("r", encoding="utf-8") as handle:
        bundle[name] = json.load(handle)

manifest = bundle["frontend_bundle_manifest.json"]
manifest_files = set(manifest.get("bundle_files", []))
expected_files = set(EXPECTED_BUNDLE_FILES)

if manifest_files != expected_files:
    raise ValueError({
        "manifest_only": sorted(manifest_files - expected_files),
        "expected_only": sorted(expected_files - manifest_files),
    })

print("bundle_file_count:", len(bundle))
print("manifest_bundle_artifact_count:", manifest.get("bundle_artifact_count"))
print("manifest_source_artifact_count:", manifest.get("source_artifact_count"))
print("manifest_lists_all_expected_files:", manifest_files == expected_files)


## 4. Detection overview summary

The next cell summarizes `detection_overview.json`. This provides the high-level validation split counts for presentation without dumping raw JSON.

Expected input: `detection_overview.json`.

Expected output: image count, detection/no-detection counts, bbox count, gallery sample count, and review status. Failure means the overview payload is missing expected frontend fields.


In [ ]:
overview = bundle["detection_overview.json"]
overview_summary = {
    "artifact_type": overview.get("artifact_type"),
    "track_id": overview.get("track_id"),
    "task_type": overview.get("task_type"),
    "run_id": overview.get("run_id"),
    "model_version": overview.get("model_version"),
    "dataset_id": overview.get("dataset_id"),
    "split": overview.get("split"),
    "image_count": overview.get("image_count"),
    "image_with_detections_count": overview.get("image_with_detections_count"),
    "image_without_detections_count": overview.get("image_without_detections_count"),
    "total_bbox_count": overview.get("total_bbox_count"),
    "gallery_sample_count": overview.get("gallery_sample_count"),
    "review_status": overview.get("review_status"),
}
for key, value in overview_summary.items():
    print(f"{key}: {value}")


## 5. Model metadata and safe status

The next cell summarizes `detection_model_metadata.json` and verifies the safe status flags used in the frontend data contract.

Expected input: `detection_model_metadata.json`.

Expected output: model/run metadata plus `production_ready=false`, `deployment_candidate=false`, and `review_required=true`. Failure means the bundle does not preserve required safety boundaries.


In [ ]:
metadata = bundle["detection_model_metadata.json"]

if metadata.get("production_ready") is not False:
    raise ValueError("Expected production_ready=false in detection_model_metadata.json")
if metadata.get("deployment_candidate") is not False:
    raise ValueError("Expected deployment_candidate=false in detection_model_metadata.json")
if metadata.get("review_required") is not True:
    raise ValueError("Expected review_required=true in detection_model_metadata.json")

metadata_summary = {
    "run_id": metadata.get("run_id"),
    "run_config_id": metadata.get("run_config_id"),
    "model_name": metadata.get("model_name"),
    "model_type": metadata.get("model_type"),
    "model_version": metadata.get("model_version"),
    "dataset_id": metadata.get("dataset_id"),
    "dataset_version": metadata.get("dataset_version"),
    "split": metadata.get("split"),
    "production_ready": metadata.get("production_ready"),
    "deployment_candidate": metadata.get("deployment_candidate"),
    "review_required": metadata.get("review_required"),
}
for key, value in metadata_summary.items():
    print(f"{key}: {value}")


## 6. Metric cards summary

The next cell summarizes `detection_metric_cards.json`. These cards are frontend-ready evidence fields for a future dashboard, not a real UI.

Expected input: `detection_metric_cards.json`.

Expected output: compact card labels, values, units, and statuses. Failure means the metric-card payload does not match the expected presentation schema.


In [ ]:
metric_cards = bundle["detection_metric_cards.json"].get("cards", [])
print("metric_card_count:", len(metric_cards))
for card in metric_cards:
    print(
        f"{card.get('card_id')}: {card.get('label')} = "
        f"{card.get('value')} {card.get('unit')} [{card.get('severity_or_status')}]"
    )


## 7. Confidence distribution summary

The next cell summarizes `detection_confidence_chart.json` and validates that confidence bin counts sum to the total bbox count.

Expected input: `detection_confidence_chart.json` and the overview total bbox count.

Expected output: confidence bin counts and global confidence summary. Failure means the confidence chart payload is inconsistent with the Detection overview.


In [ ]:
confidence = bundle["detection_confidence_chart.json"]
confidence_bins = confidence.get("confidence_bins", [])
confidence_total = sum(item.get("count", 0) for item in confidence_bins)
expected_bbox_count = overview.get("total_bbox_count")

if confidence_total != expected_bbox_count:
    raise ValueError(f"Confidence bin total {confidence_total} != total_bbox_count {expected_bbox_count}")

print("chart_title:", confidence.get("chart_title"))
print("confidence_bin_total:", confidence_total)
print("global_confidence_summary:", confidence.get("global_confidence_summary"))
for item in confidence_bins:
    print(f"{item.get('bin_label')}: {item.get('count')}")


## 8. Class summary

The next cell summarizes `detection_class_summary.json` and validates that class bbox counts sum to the total bbox count.

Expected input: `detection_class_summary.json` and the overview total bbox count.

Expected output: class count, total bbox count, and compact per-class rows. Failure means the class summary is inconsistent with the Detection overview.


In [ ]:
class_summary = bundle["detection_class_summary.json"]
class_rows = class_summary.get("class_rows", [])
class_total = sum(row.get("bbox_count", 0) for row in class_rows)

if class_total != overview.get("total_bbox_count"):
    raise ValueError(f"Class bbox total {class_total} != total_bbox_count {overview.get('total_bbox_count')}")

print("class_count:", class_summary.get("class_count"))
print("total_bbox_count:", class_summary.get("total_bbox_count"))
for row in class_rows:
    print(
        f"{row.get('class_name', row.get('class_id'))}: "
        f"bbox_count={row.get('bbox_count')}, "
        f"mean_confidence={row.get('mean_confidence')}"
    )


## 9. Sample gallery summary

The next cell summarizes `detection_sample_gallery.json`. It does not load images or create annotated images; it only presents existing sample metadata from the frontend data contract.

Expected input: `detection_sample_gallery.json`.

Expected output: gallery sample count, category IDs, and category sample counts. Failure means the gallery data contract is incomplete or malformed.


In [ ]:
gallery = bundle["detection_sample_gallery.json"]
print("gallery_sample_count:", gallery.get("gallery_sample_count"))
print("category_ids:", gallery.get("category_ids"))
print("category_sample_counts:")
for key, value in gallery.get("category_sample_counts", {}).items():
    print(f"  {key}: {value}")


## 10. Artifact lineage summary

The next cell summarizes `detection_artifact_lineage.json`. It shows which governed source artifacts and registry entries support the frontend data contract.

Expected input: `detection_artifact_lineage.json`.

Expected output: source artifacts, registry entries, hashes, and sizes. Failure means the lineage payload is missing governance traceability fields.


In [ ]:
lineage = bundle["detection_artifact_lineage.json"]
print("source_artifacts:")
for item in lineage.get("source_artifacts", []):
    print(f"  - {item}")
print("registry_entries:")
for item in lineage.get("registry_entries", []):
    print(f"  - {item}")
print("artifact_hashes:", lineage.get("artifact_hashes"))
print("artifact_size_bytes:", lineage.get("artifact_size_bytes"))


## 11. Quality decision and frontend recommendation

The next cell summarizes `detection_quality_decision_summary.json` and `frontend_detection_recommendation.json`. It verifies safe status flags and presents the recommended next step.

Expected input: quality decision and frontend recommendation JSON files.

Expected output: review-required decision, false production/deployment flags, and a safe recommendation. Failure means the bundle violates expected safety boundaries.


In [ ]:
quality = bundle["detection_quality_decision_summary.json"]
recommendation = bundle["frontend_detection_recommendation.json"]

if quality.get("production_ready") is not False:
    raise ValueError("Expected production_ready=false in detection_quality_decision_summary.json")
if quality.get("deployment_candidate") is not False:
    raise ValueError("Expected deployment_candidate=false in detection_quality_decision_summary.json")
if quality.get("review_required") is not True:
    raise ValueError("Expected review_required=true in detection_quality_decision_summary.json")

print("decision:", quality.get("decision"))
print("review_required:", quality.get("review_required"))
print("production_ready:", quality.get("production_ready"))
print("deployment_candidate:", quality.get("deployment_candidate"))
print("next_recommended_step:", quality.get("next_recommended_step"))
print("recommendation_status:", recommendation.get("recommendation_status"))
print("frontend_next_step:", recommendation.get("next_step"))
print("what_it_can_claim:", recommendation.get("what_it_can_claim"))
print("what_it_cannot_claim:", recommendation.get("what_it_cannot_claim"))


## 12. Limitations and safe boundaries

The next cell prints the explicit limitations and safe boundaries from the frontend data contract. This makes the review position clear without introducing new claims.

Expected input: limitation fields from overview, lineage, quality decision, recommendation, and manifest payloads.

Expected output: concise limitation statements. Failure would mean a bundle payload is missing expected limitation fields, but it would not affect source artifacts.


In [ ]:
limitation_sources = {
    "overview_limitations": overview.get("limitations", []),
    "lineage_limitations": lineage.get("limitations", []),
    "quality_limitations": quality.get("limitations", []),
    "recommendation_cannot_claim": recommendation.get("what_it_cannot_claim", []),
    "manifest_safe_demo_wording": manifest.get("safe_demo_wording"),
}

for section, values in limitation_sources.items():
    print(section + ":")
    if isinstance(values, list):
        for value in values:
            print(f"  - {value}")
    else:
        print(f"  - {values}")

print("safe_boundary: no production-ready claim")
print("safe_boundary: no deployment-safe claim")
print("safe_boundary: not real frontend/UI")
print("safe_boundary: not API")
print("safe_boundary: not deployment approval")


## 13. Final interpretation and next step

The next cell provides the final reviewer-facing summary. It uses only values loaded from the existing frontend data-contract bundle.

Expected input: validated bundle payloads from previous sections.

Expected output: final status line and safe next-step statement. Failure means a prior validation failed and the notebook should not be used for presentation until the bundle is checked.


In [ ]:
final_summary = {
    "YOLO / Detection evidence layer": "COMPLETE",
    "image_count": overview.get("image_count"),
    "total_bbox_count": overview.get("total_bbox_count"),
    "gallery_sample_count": gallery.get("gallery_sample_count"),
    "production_ready": quality.get("production_ready"),
    "deployment_candidate": quality.get("deployment_candidate"),
    "review_required": quality.get("review_required"),
    "not_real_frontend": True,
    "not_api": True,
    "not_deployment_approval": True,
}

for key, value in final_summary.items():
    print(f"{key}: {value}")

print("final_interpretation: Detection frontend data-contract evidence is ready for presentation.")
print("next_step: use this notebook alongside the final demo packaging docs; do not train, recompute metrics, update registries, or claim deployment approval from this notebook.")
